First things first, lets figure out how to use Polymarket API to get probabilities

First, I need to find the tag(s) associated with NCAABB

In [1]:
import requests

# Step 1: Find the tag ID for NCAA Basketball
tags_response = requests.get("https://gamma-api.polymarket.com/sports")
tags = tags_response.json()



In [49]:
tags

[{'id': 1,
  'sport': 'ncaab',
  'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/marchmadness.jpeg',
  'resolution': 'https://www.ncaa.com/march-madness-live/bracket',
  'ordering': 'home',
  'tags': '1,100149,100639',
  'series': '39',
  'createdAt': '2025-11-05T19:27:45.399303Z'},
 {'id': 2,
  'sport': 'epl',
  'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/Repetitive-markets/premier+league.jpg',
  'resolution': 'https://www.premierleague.com/',
  'ordering': 'home',
  'tags': '1,82,306,100639,100350',
  'series': '10188',
  'createdAt': '2025-11-05T19:27:45.399303Z'},
 {'id': 3,
  'sport': 'lal',
  'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/league-lal.png',
  'resolution': 'https://www.laliga.com/en-GB',
  'ordering': 'home',
  'tags': '1,780,100639,100350',
  'series': '10193',
  'createdAt': '2025-11-05T19:27:45.399303Z'},
 {'id': 95,
  'sport': 'acn',
  'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/africa-cup-of-

Okay, it looks like the tags are 1,100149,100639, but only relevant one is 100149 

In [75]:
# Step 2: Use the tag_id to fetch related events
tag_id = 100149
events_response = requests.get(
    "https://gamma-api.polymarket.com/events",
    params={
        "tag_id": tag_id,
        "active": "true",
        "closed": "false",
        "limit": 100
    }
)
events = events_response.json()

# Step 3: Extract individual markets from events
relevant_ids = []
for event in events:
    print(f"Event: {event.get('title')}")
    for market in event.get("markets", []):
        question = market.get('question')
        print(f"  Market: {question}")
        if "(" not in question and "O/U" not in question:
            relevant_ids.append(market["id"])
            

Event: Will there be a perfect NCAA bracket?
  Market: Will there be a perfect NCAA bracket?
Event: Illinois Fighting Illini vs. Houston Cougars
  Market: Illinois Fighting Illini vs. Houston Cougars
  Market: Spread: Houston Cougars (-3.5)
  Market: Illinois Fighting Illini vs. Houston Cougars: O/U 139.5
  Market: Illinois Fighting Illini vs. Houston Cougars: O/U 140.5
  Market: Spread: Houston Cougars (-2.5)
Event: Texas Longhorns vs. Purdue Boilermakers
  Market: Texas Longhorns vs. Purdue Boilermakers
  Market: Spread: Purdue Boilermakers (-6.5)
  Market: Texas Longhorns vs. Purdue Boilermakers: O/U 148.5
  Market: Spread: Purdue Boilermakers (-7.5)
  Market: Texas Longhorns vs. Purdue Boilermakers: O/U 147.5
Event: St. John's Red Storm vs. Duke Blue Devils
  Market: St. John's Red Storm vs. Duke Blue Devils
  Market: Spread: Duke Blue Devils (-6.5)
  Market: St. John's Red Storm vs. Duke Blue Devils: O/U 141.5
Event: Tennessee Volunteers vs. Iowa State Cyclones
  Market: Tennessee

Great, now that we have our markets, lets pull some data for one

In [76]:
relevant_ids

['1646141',
 '1678319',
 '1684991',
 '1686946',
 '1687407',
 '1687801',
 '1688000',
 '1688392',
 '1688876']

In [60]:
market_id

'1719237'

In [77]:
market_id = relevant_ids[-1]

import requests

url = f"https://gamma-api.polymarket.com/markets/{market_id}"

response = requests.get(url).json()


In [78]:
response['outcomes']

'["Alabama Crimson Tide", "Michigan Wolverines"]'

In [79]:
response['outcomePrices']

'["0.185", "0.815"]'

Alright so we got basic game probabilities, but we really want some prop bets as well (probability michigan wins chip, etc)

So it looks like they have futures on these exact things Im looking for. The way I found it was to plug in the suffix of the web address for the specific prop I'm looking for

In [2]:
def extract_team(question):
    return question.split("Will ")[1].split(" advance to")[0]

In [3]:
# quarter finals first

import requests

slug = "ncaa-tournament-team-to-make-quarterfinals"  # from the URL
response = requests.get(f"https://gamma-api.polymarket.com/events/slug/{slug}")
event = response.json()

qf_map = {}
for market in event.get("markets", []):
    if (not market["closed"]) and market['active']:
        question = market.get('question')
#         print(f"Market ID: {market.get('id')}, Question: {question}")
        bid = float(market["bestBid"])
        ask = float(market["bestAsk"])
        mid = (bid + ask) / 2
#         print("Implied probability:", mid, '\n')
        team = extract_team(question)
        qf_map[team] = mid
        
qf_map

KeyError: 'bestBid'

In [3]:
# then semis

import requests

slug = "ncaa-tournament-team-to-make-semifinals"  # from the URL
response = requests.get(f"https://gamma-api.polymarket.com/events/slug/{slug}")
event = response.json()

sf_map = {}
for market in event.get("markets", []):
    if (not market["closed"]) and market['active']:
        question = market.get('question')
#         print(f"Market ID: {market.get('id')}, Question: {question}")
        bid = float(market["bestBid"])
        ask = float(market["bestAsk"])
        mid = (bid + ask) / 2
#         print("Implied probability:", mid, '\n')
        team = extract_team(question)
        sf_map[team] = mid

sf_map

{'Arizona': 0.7,
 'UConn': 0.195,
 'Illinois': 0.745,
 'Iowa State': 0.24,
 'Alabama': 0.0695,
 'Tennessee': 0.125,
 'Duke': 0.485,
 'Michigan': 0.5700000000000001,
 'Michigan State': 0.12000000000000001,
 'St. John’s': 0.115,
 'Purdue': 0.305,
 'Iowa': 0.27249999999999996}

In [4]:
# then make the finals

import requests

slug = "ncaa-tournament-team-to-make-national-championship"  # from the URL
response = requests.get(f"https://gamma-api.polymarket.com/events/slug/{slug}")
event = response.json()


f_map = {}
for market in event.get("markets", []):
    if (not market["closed"]) and market['active']:
        question = market.get('question')
        if "Team" not in question:
#             print(f"Market ID: {market.get('id')}, Question: {question}")
            bid = float(market["bestBid"])
            ask = float(market["bestAsk"])
            mid = (bid + ask) / 2
#             print("Implied probability:", mid, '\n')
            team = extract_team(question)
            f_map[team] = mid
        
f_map

{'Arizona': 0.41500000000000004,
 'Purdue': 0.3,
 'Tennessee': 0.223,
 'Illinois': 0.355,
 'Duke': 0.28,
 'UConn': 0.21,
 'Iowa': 0.2225,
 'Michigan': 0.245,
 'Michigan State': 0.27,
 'Iowa State': 0.27999999999999997,
 'Alabama': 0.030000000000000002,
 'St. John’s': 0.06}

In [5]:
# and finally, win the chip

import requests

slug = "2026-ncaa-tournament-winner"  # from the URL
response = requests.get(f"https://gamma-api.polymarket.com/events/slug/{slug}")
event = response.json()

chip_map = {}
for market in event.get("markets", []):
    if (not market["closed"]) and market['active']:
        question = market.get('question')
        if "Team" not in question:
#             print(f"Market ID: {market.get('id')}, Question: {question}")
            bid = float(market["bestBid"])
            ask = float(market["bestAsk"])
            mid = (bid + ask) / 2
#             print("Implied probability:", mid, '\n')
            team = question.split("Will ")[1].split(" win")[0]
            chip_map[team] = mid

chip_map

{'Michigan': 0.20500000000000002,
 'Alabama': 0.0085,
 'Michigan State': 0.0245,
 'Arizona': 0.2715,
 'Connecticut': 0.0345,
 'Iowa State': 0.048,
 'Purdue': 0.0625,
 'Duke': 0.175,
 'Illinois': 0.1255,
 'Tennessee': 0.0155,
 "St. John's": 0.037500000000000006,
 'Iowa': 0.0155}

Now that we have those, we can go about making our probability function and therefore estimating expected value!

However first, must reformat into an easy to ingest format.

For now, lets go with a dataframe

In [14]:
import pandas as pd

# def concatenate_data(qf_map, sf_map, f_map, chip_map):

#     common_teams = set(qf_map) & set(sf_map) & set(f_map) & set(chip_map)

#     qf_map_filt   = {k: qf_map[k] for k in common_teams}
#     sf_map_filt   = {k: sf_map[k] for k in common_teams}
#     f_map_filt    = {k: f_map[k] for k in common_teams}
#     chip_map_filt = {k: chip_map[k] for k in common_teams}

#     qf_map = dict(sorted(qf_map_filt.items()))
#     sf_map = dict(sorted(sf_map_filt.items()))
#     f_map = dict(sorted(f_map_filt.items()))
#     chip_map = dict(sorted(chip_map_filt.items()))

#     import pandas as pd

#     df = pd.DataFrame({
#         "team": qf_map.keys(),
#         "quarter_finals": qf_map.values(),
#         "semi_finals": sf_map.values(),
#         "finals": f_map.values(),
#         "championship": chip_map.values()
#     })
    
#     return df

def concatenate_data(qf_map, sf_map, f_map, chip_map):
    import pandas as pd

    # Use sf_map as the base universe
    teams = sorted(sf_map.keys())

    df = pd.DataFrame({
        "team": teams,
        "quarter_finals": [qf_map.get(t, 0) if t in qf_map else 1 for t in teams],
        "semi_finals": [sf_map.get(t, 0) for t in teams],
        "finals": [f_map.get(t, 0) for t in teams],
        "championship": [chip_map.get(t, 0) for t in teams]
    })

    return df

df = concatenate_data(qf_map, sf_map, f_map, chip_map)

df

,team,quarter_finals,semi_finals,finals,championship
0,Alabama,0.195,0.0695,0.0300,0.0085
1,Arizona,1.000,0.7000,0.4150,0.2715
2,Duke,0.710,0.4850,0.2800,0.1750
3,Illinois,1.000,0.7450,0.3550,0.1255
4,Iowa,1.000,0.2725,0.2225,0.0155
5,Iowa State,0.675,0.2400,0.2800,0.0480
6,Michigan,0.920,0.5700,0.2450,0.2050
7,Michigan State,0.255,0.1200,0.2700,0.0245
8,Purdue,1.000,0.3050,0.3000,0.0625
9,St. John’s,0.300,0.1150,0.0600,0.0000


In [7]:
# test to make sure dataframe is properly constructed

for key in qf_map:
    assert df.loc[(df.team == key)]['quarter_finals'].item() == qf_map[key]
    
for key in sf_map:
    assert df.loc[(df.team == key)]['semi_finals'].item() == sf_map[key]
        
for key in f_map:
    assert df.loc[(df.team == key)]['finals'].item() == f_map[key]
    
for key in chip_map:
    assert df.loc[(df.team == key)]['championship'].item() == chip_map[key]

AssertionError: 

Now, lets take a step back and remember what we have. 

We have 

<div style="text-align: center;">
$P_{qf}$ = P(team advances to quarter-finals) <br>
$P_{sf}$ = P(team advances to semi-finals) <br>
$P_{f}$ = P(team advances to finals) <br>
$P_{c}$ = P(team wins championship) <br>
</div>

Recall the payoff structure of the future:

<center>
64 for champion <br>
32 for runner up <br>
16 for losing in the Final 4 Round <br>
8 for losing in the Elite 8 Round <br>
4 for losing in the Sweet 16 Round <br>
2 for losing in the Second Round <br>
0 for losing in the First Round <br>
0 for losing in the First Four <br>
</center>

So therefore 

<center>
$P_{S16}$ = P(lose in sweet 16) = 1 - $P_{qf}$ <br>
</center>

Now is the interesting part. Obviously the probability a team advances to the semi-finals is dependent on advancing to quarterfinals and winning them.

Ie
<center>
$P_{sf}$ = P(win qf game | team advances to qf) $P_{qf}$.
</center>

Therefore

<center>
P(win qf game | team advances to qf) = $\frac{P_{sf}}{P_{qf}}$
</center>

and thus we can back out the probability of losing in the final 4 round;

<center>
P(lose in qf) = P(lose qf game | team advances to qf) * $P_{qf}$ <br>
= (1 - $\frac{P_{sf}}{P_{qf}}) * {P_{qf}}$ <br>
= $P_{qf}$ - $P_{sf}$
</center>

We can extend this to find

<center>
    P(lose in sf) = $P_{sf}$ - $P_{f}$ <br>
    and <br>
    P(lose in finals) = $P_{f}$ - $P_{c}$ <br>
</center>
    

In [21]:
# however, before we do anything, need to smooth the probabilities
# and enforce monotonicity, but in the simplest way possible

# these are weird inconsistencies that can be arb'd tho so maybe keep that in mind

import numpy as np

cols = ["quarter_finals", "semi_finals", "finals", "championship"]


def smooth_with_min_gap(row):
    vals = row[cols].values.copy()
    
    eps = 0.02 * vals[0]  # 2% of QF probability
    
    # Step 1: enforce monotonicity + minimum gaps
    for i in range(1, len(vals)):
        vals[i] = min(vals[i], vals[i-1] - eps)
    
    # Step 2: keep valid range
    vals = np.clip(vals, 0, 1)
    
    return vals

df[cols] = np.vstack(df.apply(smooth_with_min_gap, axis=1))

# test to ensure it works
assert (df["quarter_finals"] > df["semi_finals"]).all()
assert (df["semi_finals"] > df["finals"]).all()
assert (df["finals"] > df["championship"]).all()

df

,team,quarter_finals,semi_finals,finals,championship
0,Alabama,0.195,0.0695,0.03,0.0085
1,Arizona,1.0,0.7,0.415,0.2715
2,Duke,0.71,0.485,0.28,0.175
3,Illinois,1.0,0.745,0.355,0.1255
4,Iowa,1.0,0.2725,0.2225,0.0155
5,Iowa State,0.675,0.24,0.2265,0.048
6,Michigan,0.92,0.57,0.245,0.205
7,Michigan State,0.255,0.12,0.1149,0.0245
8,Purdue,1.0,0.305,0.285,0.0625
9,St. John’s,0.3,0.115,0.06,0.0


In [101]:
# compute p(losing in particular round)
df["lose_in_qf"] = df["quarter_finals"] - df["semi_finals"]
df["lose_in_sf"] = df["semi_finals"] - df["finals"]
df["lose_in_finals"] = df["finals"] - df["championship"]


df["total_prob"] = (
    (1 - df["quarter_finals"]) +   # lose before QF
    df["lose_in_qf"] +
    df["lose_in_sf"] +
    df["lose_in_finals"] +
    df["championship"]
)

# ensure probs sum to 1
assert (df["total_prob"] - 1 < 1e-13).all()

df

,team,quarter_finals,semi_finals,finals,championship,lose_in_qf,lose_in_sf,lose_in_finals,total_prob
0,Alabama,0.145,0.078,0.034,0.0075,0.067,0.044,0.0265,1.0
1,Arizona,0.445,0.4361,0.315,0.197,0.0089,0.1211,0.118,1.0
2,Arkansas,0.235,0.1,0.0435,0.0175,0.135,0.0565,0.026,1.0
3,Duke,0.77,0.495,0.325,0.175,0.275,0.17,0.15,1.0
4,Houston,0.68,0.43,0.22,0.105,0.25,0.21,0.115,1.0
5,Illinois,0.415,0.315,0.16,0.0625,0.1,0.155,0.0975,1.0
6,Iowa,0.455,0.0875,0.0784,0.011,0.3675,0.0091,0.0674,1.0
7,Iowa State,0.64,0.25,0.115,0.054,0.39,0.135,0.061,1.0
8,Michigan,0.815,0.59,0.23,0.2137,0.225,0.36,0.0163,1.0
9,Michigan State,0.39,0.165,0.065,0.023,0.225,0.1,0.042,1.0


And now, after all that hard work, the most critical calculation, expected value

In [102]:
df["EV"] = (
    4 * (1 - df["quarter_finals"]) +
    8 * df["lose_in_qf"] +
    16 * df["lose_in_sf"] +
    32 * df["lose_in_finals"] +
    64 * df["championship"]
)

df

,team,quarter_finals,semi_finals,finals,championship,lose_in_qf,lose_in_sf,lose_in_finals,total_prob,EV
0,Alabama,0.145,0.078,0.034,0.0075,0.067,0.044,0.0265,1.0,5.988
1,Arizona,0.445,0.4361,0.315,0.197,0.0089,0.1211,0.118,1.0,20.6128
2,Arkansas,0.235,0.1,0.0435,0.0175,0.135,0.0565,0.026,1.0,6.996
3,Duke,0.77,0.495,0.325,0.175,0.275,0.17,0.15,1.0,21.84
4,Houston,0.68,0.43,0.22,0.105,0.25,0.21,0.115,1.0,17.04
5,Illinois,0.415,0.315,0.16,0.0625,0.1,0.155,0.0975,1.0,12.74
6,Iowa,0.455,0.0875,0.0784,0.011,0.3675,0.0091,0.0674,1.0,8.1264
7,Iowa State,0.64,0.25,0.115,0.054,0.39,0.135,0.061,1.0,12.128
8,Michigan,0.815,0.59,0.23,0.2137,0.225,0.36,0.0163,1.0,22.4984
9,Michigan State,0.39,0.165,0.065,0.023,0.225,0.1,0.042,1.0,8.656


While rudimentary, its a nice start. Time to go back and think thru strategy now. 

In acknowledging there will be people better at estimating expected value of these contracts, it then becomes imperative to execute a simple strategy intelligently, not just relying on correctness of expected value estimates.